# Data Ingestion Runbook

This notebook documents the KB data-ingestion workflow for preparing metadata, converting metadata to UTF-8-compatible format, and ingesting non-multimodal documents into Elasticsearch.

## Workflow
1. Set the working directory and import dependencies.
2. Configure required environment variables.
3. Download documents from SharePoint and upload them to CDSW while maintaining the SharePoint folder structure.
4. Create the initial metadata table.
5. Manually amend the semi-automated metadata table.
6. Convert the completed metadata table to UTF-8-compatible format.
7. Create document chunks and ingest them into Elasticsearch.

In [ ]:
import os

import pandas as pd

from srs.metadata import create_metadata, convert_metadata_df_to_utf8_compatible
from sre.non_multimodal_processing import create_chunks_from_documents
from src.elasticsearch import get_docs_to_reingest, ingest
from elk_utils import connect_to_elk, delete_index
from ada_genai.auth import ssoauth


## 0. Set environment variables

Set the following environment variables in either **Project Settings** or **Account Settings**.

| Key | Example / Value |
|---|---|
| `ONE_BANK_ID` | Your One Bank ID, e.g. `B,1bankid` |
| `ONE_BANK_PASSWORD` | Your One Bank password, e.g. `x0ox` |
| `SHAREPOINT_URL` | e.g. `https://dbs1bank.sharepoint.com/sites/sgibg_kb/Shared%20Documents/` |
| `CDSW_SHAREPOINT_PATH` | e.g. `/home/cdsw/data/COLLECTION_NAME` |
| `COLLECTION_NAME` | e.g. `datasets-kb.kb-ibg.sg.uat` |

> **Important:** Do not hard-code credentials in the notebook. Store them in the appropriate Project/Account Settings.

In [ ]:
# Set the working directory for the data-ingestion project.
os.chdir("/home/cdsw/ibekb-data-ingestion")

# Authenticate with SSO.
ssoauth.login()


## 1. Download documents from SharePoint and upload to CDSW

Download the required documents from SharePoint and upload them to CDSW.

**Maintain the same folder structure in CDSW as in SharePoint.**

The source runbook indicates that the local SharePoint path is provided through `CDSW_SHAREPOINT_PATH`.

In [ ]:
sharepoint_path = os.environ["CDSW_SHAREPOINT_PATH"]
collection_name = os.environ["COLLECTION_NAME"]

print(f"Collection: {collection_name}")
print(f"CDSW SharePoint path: {sharepoint_path}")


## 2.1 Create metadata table

Generate the initial metadata table from the downloaded document directory.

In [ ]:
batch = "batch2"

metadata_dir = f"/home/cdsw/metadata/{batch}"
metadata_df_path = f"{metadata_dir}/metadata.csv"

os.makedirs(metadata_dir, exist_ok=True)

metadata_df = create_metadata(
    file_directory=os.environ["CDSW_SHAREPOINT_PATH"]
)

# Overwrite the output file if it already exists.
metadata_df.to_csv(metadata_df_path, index=False)

metadata_df


## 2.2 Manually amend the semi-automated metadata table

Open the generated `metadata.csv` and manually complete the metadata fields.

### Document name
- Create a new column: `document name`.
- Identify the most appropriate document name based on the filename and document content.
- Delete the intermediate columns `document name based on filename` and `document name based on first page` after the final document name has been determined.

### Date of issue
- Create a new column: `date of issue`.
- Determine the appropriate date of issue using the available metadata/content.
- Delete the intermediate columns `date of issue based on filename` and `date of issue based on first page` after the final date of issue has been determined.

> **Manual step:** The source runbook explicitly requires human review and amendment of these fields.

## 2.3 Convert metadata table to UTF-8-compatible format

After manually completing the metadata table, save it as `metadata_fill_in_new.csv` and convert it to a UTF-8-compatible CSV.

In [ ]:
metadata_fill_in_path = f"{metadata_dir}/metadata_fill_in_new.csv"
metadata_fill_in_utf8_path = f"{metadata_dir}/metadata_fill_in_utf8.csv"

convert_metadata_df_to_utf8_compatible(
    metadata_fill_in_path=metadata_fill_in_path,
    metadata_fill_in_utf8_path=metadata_fill_in_utf8_path,
)

metadata_fill_in_df = pd.read_csv(metadata_fill_in_utf8_path)
metadata_fill_in_df


## 3. Data ingestion — non-multimodal

Connect to Elasticsearch, create document chunks, and ingest the documents.

In [ ]:
# Connect to Elasticsearch for the selected collection.
collection_name = os.environ["COLLECTION_NAME"]
es_store_instance, es_instance = connect_to_elk(collection_name)


### 3.1 Create document chunks

Create chunks from the documents using the completed UTF-8-compatible metadata.

In [ ]:
all_document_chunk_list = create_chunks_from_documents(
    metadata_df=metadata_fill_in_df,
    file_directory=os.environ["CDSW_SHAREPOINT_PATH"],
)

print(f"Total document chunks: {len(all_document_chunk_list)}")


### 3.2 Ingest documents

Ingest the generated document chunks into Elasticsearch.

The source runbook contains two ingestion configurations. The first uses `batch_size=100`; the second uses `batch_size=30` and is retained below as a separate option.

In [ ]:
# Option A: standard ingestion configuration from the source runbook.
docs_to_reingest = ingest(
    all_document_chunk_list=all_document_chunk_list,
    es_store_instance=es_store_instance,
    es_instance=es_instance,
    collection_name=collection_name,
    batch_size=100,
    timer_per_document=30,
)

docs_to_reingest


### 3.3 Alternative ingestion configuration

Use this configuration only when the smaller batch size is required.

In [ ]:
# Option B: smaller ingestion batch.
# Uncomment and run this block only if required.
#
# docs_to_reingest = ingest(
#     all_document_chunk_list=all_document_chunk_list,
#     es_store_instance=es_store_instance,
#     es_instance=es_instance,
#     collection_name=collection_name,
#     batch_size=30,
#     timer_per_document=30,
# )


## 4. Re-ingestion / index maintenance

The source notebook imports `get_docs_to_reingest` and `delete_index`, but the pasted source does not contain enough information to reconstruct their intended invocation safely.

Validate the project-specific function signatures and the intended re-ingestion/index-deletion sequence before adding production execution cells.

In [ ]:
# Available utilities imported from the source runbook:
# get_docs_to_reingest(...)
# delete_index(...)
#
# Add the project-specific invocation here after validating their APIs.
